# MiniMax H3 on Kaggle — Wan2GP + Turbo LoRA

Designed for **Kaggle 2×T4**. This notebook keeps the large H3 model and Turbo LoRA in `/tmp` to avoid filling `/kaggle/working`.

**Target test:** MiniMax H3 FL2VA Pruned 20B, 480p, ~5 seconds, Turbo v4 Step-600 EMA, 6 steps.


## 1. Check GPU and disk


In [ ]:
!nvidia-smi

import torch, os
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), 
          f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

## 2. Clone Wan2GP


In [ ]:
%cd /kaggle/working
import os
if not os.path.exists('/kaggle/working/Wan2GP'):
    !git clone https://github.com/deepbeepmeep/Wan2GP.git
%cd /kaggle/working/Wan2GP


## 3. Install Wan2GP dependencies

In [ ]:
%cd /kaggle/working/Wan2GP

!pip install -r requirements.txt

## 4. Create temporary model

In [ ]:
!mkdir -p /tmp/minimax_h3
!df -h /tmp

## 5. Download the H3 pruned INT8 ConvRot base

This is about 21 GB, so it goes to `/tmp`.

In [ ]:
from huggingface_hub import hf_hub_download
import os

base = hf_hub_download(
    repo_id='DeepBeepMeep/MiniMax-H3',
    filename='MiniMax-H3-FL2VA-pruned_int8_convrot.safetensors',
    local_dir='/tmp/minimax_h3'
)
print('Base:', base)
!ls -lh /tmp/minimax_h3


## 6. Download the Qwen3-VL text encoder

Use the actual file discovered in the MiniMax-H3 repository.

In [5]:
encoder = hf_hub_download(
    repo_id='DeepBeepMeep/MiniMax-H3',
    filename='Qwen3-VL-32B-Instruct/Qwen3-VL-32B-Instruct-layer50_quanto_bf16_int8.safetensors',
    local_dir='/tmp/minimax_h3/Qwen3-VL-32B-Instruct'
)
print('Encoder:', encoder)
!du -sh /tmp/minimax_h3


## 7. Launch Wan2GP

The current Wan2GP build exposes `--lora-dir-minimax-h3`, so the LoRA can stay in `/tmp` instead of consuming working-disk space.

**Run this cell last.** It will keep the notebook cell occupied while Wan2GP is running. Open the Gradio URL printed by the cell.

In [ ]:
%cd /kaggle/working/Wan2GP

!python wgp.py --share

## 7. URLs

For your MiniMax H3 setup:

### Main Checkpoints

```text
/tmp/minimax_h3/MiniMax-H3-FL2VA-pruned_int8_convrot.safetensors
```

### Text Encoder Checkpoints

```text
/tmp/minimax_h3/Qwen3-VL-32B-Instruct/Qwen3-VL-32B-Instruct/Qwen3-VL-32B-Instruct-layer50_quanto_bf16_int8.safetensors
```

### Video VAE File

Keep the existing MiniMax H3 **Video VAE** entry from the source model.

### Audio VAE File

Keep the existing MiniMax H3 **Audio VAE** entry from the source model.



## 8. Suggested first-generation settings

- Model: **MiniMax H3 / FL2VA Pruned 20B**
- Turbo LoRA: **Turbo EMA ckpt850 4 steps**
- LoRA strength: **1.0**
- Steps: **6**
- Scheduler: **simple** if exposed by the H3 Turbo workflow
- Resolution: **480p**
- Duration: **~5 seconds / 124 frames**
- Keep other acceleration methods OFF for the first test.

**Note:** The Turbo LoRA's README describes a dedicated MiniMax-H3 Turbo Sampler for the intended 4–8-step workflow. Wan2GP's H3 LoRA directory support confirms it can discover the LoRA, but this notebook does not assume Wan2GP's normal sampler is identical to that dedicated ComfyUI sampler.

## 9. GPU monitoring (run in a separate cell after starting generation)


In [ ]:
!nvidia-smi